In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GraphAttention(nn.Module):
    def __init__(self, in_features, out_features, n_heads=4, dropout=0.1, alpha=0.2):
        super(GraphAttention, self).__init__()
        self.in_features = in_features
        self.out_features = out_features  # out_features_per_head
        self.n_heads = n_heads
        self.dropout = dropout
        self.alpha = alpha

        # Linear transformations
        self.W = nn.Linear(in_features, n_heads * out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)

        # Initialize parameters
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a.weight)

        self.leakyrelu = nn.LeakyReLU(self.alpha)
        self.dropout_layer = nn.Dropout(self.dropout)

    def forward(self, h, adj):
        # h: [batch_size, num_nodes, in_features] (32, 50, 128)
        # adj: [batch_size, num_nodes, num_nodes] (32, 50, 50)

        batch_size, num_nodes = h.size(0), h.size(1)

        # Linear transformation
        Wh = self.W(h)  # [32, 50, n_heads*out_features] (32,50,4*32=128)
        Wh = Wh.view(batch_size, num_nodes, self.n_heads, self.out_features)  # [32,50,4,32]
        Wh = Wh.permute(0, 2, 1, 3)  # [32,4,50,32]

        # Compute attention coefficients
        Wh_repeated_i = Wh.unsqueeze(3).expand(-1, -1, -1, num_nodes, -1)  # [32,4,50,50,32]
        Wh_repeated_j = Wh.unsqueeze(2).expand(-1, -1, num_nodes, -1, -1)  # [32,4,50,50,32]
        concat = torch.cat([Wh_repeated_i, Wh_repeated_j], dim=-1)  # [32,4,50,50,64]

        e = self.leakyrelu(self.a(concat).squeeze(-1))  # [32,4,50,50]

        # Mask attention coefficients
        zero_vec = -9e15 * torch.ones_like(e)
        adj = adj.unsqueeze(1)  # [32,1,50,50]
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=-1)  # [32,4,50,50]
        attention = self.dropout_layer(attention)

        # Apply attention
        h_prime = torch.matmul(attention, Wh)  # [32,4,50,32]
        h_prime = h_prime.permute(0, 2, 1, 3).contiguous()  # [32,50,4,32]
        h_prime = h_prime.view(batch_size, num_nodes, -1)  # [32,50,128]

        return h_prime


class DualOutputGRAN(nn.Module):
    """
    Graph Recurrent Attention Network that generates both adjacency matrices and amino acid sequences.
    """
    def __init__(self, node_features=22, hidden_dim=128, num_layers=2,
                 n_heads=4, dropout=0.1, amino_acid_vocab_size=22):
        super(DualOutputGRAN, self).__init__()
        self.node_features = node_features
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.n_heads = n_heads
        self.amino_acid_vocab_size = amino_acid_vocab_size

        assert hidden_dim % n_heads == 0, "hidden_dim must be divisible by n_heads"
        self.out_features_per_head = hidden_dim // n_heads

        # Node feature embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # Graph attention layers
        self.gat_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.gat_layers.append(GraphAttention(
                in_features=hidden_dim,
                out_features=self.out_features_per_head,
                n_heads=n_heads,
                dropout=dropout
            ))

        # Sequence generation
        self.rnn_cell = nn.GRUCell(hidden_dim, hidden_dim)
        self.sequence_projection = nn.Linear(hidden_dim, amino_acid_vocab_size)

        # Adjacency matrix generation
        self.edge_predictor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

        self.dropout = nn.Dropout(dropout)

    def _process_graph(self, node_features, adjacency_matrix):
        """Process the input graph with graph attention layers"""
        h = self.node_embedding(node_features)  # [32,50,22] -> [32,50,128]

        for gat_layer in self.gat_layers:
            h = gat_layer(h, adjacency_matrix)  # [32,50,128]
            h = F.elu(h)
            h = self.dropout(h)

        return h

    def _generate_adjacency(self, node_embeddings):
        """Generate adjacency matrix from node embeddings"""
        batch_size, num_nodes, _ = node_embeddings.size()

        # Create all pairwise combinations of node embeddings
        node_i = node_embeddings.unsqueeze(2).repeat(1, 1, num_nodes, 1)  # [B, N, N, H]
        node_j = node_embeddings.unsqueeze(1).repeat(1, num_nodes, 1, 1)  # [B, N, N, H]

        # Concatenate node pairs
        node_pairs = torch.cat([node_i, node_j], dim=-1)  # [B, N, N, 2H]

        # Reshape for passing through edge predictor
        flat_pairs = node_pairs.view(-1, 2 * self.hidden_dim)  # [B*N*N, 2H]

        # Predict edges
        edge_scores = self.edge_predictor(flat_pairs).view(batch_size, num_nodes, num_nodes)  # [B, N, N]

        # Ensure symmetry (for undirected graphs)
        edge_scores = (edge_scores + edge_scores.transpose(1, 2)) / 2

        return edge_scores

    def forward(self, node_features, adjacency_matrix, target_sequences=None, target_adjacency=None, max_length=100):
        """
        Forward pass with dual outputs (sequence and adjacency matrix)

        Args:
            node_features: [batch_size, num_nodes, node_feature_dim]
            adjacency_matrix: [batch_size, num_nodes, num_nodes]
            target_sequences: [batch_size, seq_length] (for training)
            target_adjacency: [batch_size, num_nodes, num_nodes] (for training)
            max_length: Maximum sequence length for generation

        Returns:
            Dictionary containing:
                - 'sequence_logits': Predicted sequence logits
                - 'adjacency_matrix': Predicted adjacency matrix
                - Or generated sequence and adjacency matrix during inference
        """
        batch_size, num_nodes = node_features.size(0), node_features.size(1)

        # Process graph
        node_embeddings = self._process_graph(node_features, adjacency_matrix)  # [B, N, H]

        # Generate adjacency matrix
        predicted_adjacency = self._generate_adjacency(node_embeddings)  # [B, N, N]

        # Initialize RNN for sequence generation
        graph_embedding = torch.mean(node_embeddings, dim=1)  # [B, H]
        h_t = graph_embedding  # Initial hidden state

        if target_sequences is not None:
            # Training mode
            seq_length = target_sequences.size(1)
            sequence_logits = torch.zeros(batch_size, seq_length, self.amino_acid_vocab_size,
                                          device=node_features.device)

            x_t = torch.zeros(batch_size, self.hidden_dim, device=node_features.device)

            for t in range(seq_length):
                h_t = self.rnn_cell(x_t, h_t)  # [B, H]
                sequence_logits[:, t, :] = self.sequence_projection(h_t)  # [B, vocab_size]

                if t < seq_length - 1:
                    x_t = self.node_embedding(
                        F.one_hot(target_sequences[:, t],
                                  num_classes=self.amino_acid_vocab_size).float()
                    )  # [B, H]

            return {
                'sequence_logits': sequence_logits,
                'adjacency_matrix': predicted_adjacency
            }

        else:
            # Generation mode
            generated_sequences = torch.zeros(batch_size, max_length,
                                              dtype=torch.long,
                                              device=node_features.device)
            x_t = torch.zeros(batch_size, self.hidden_dim,
                              device=node_features.device)

            for t in range(max_length):
                h_t = self.rnn_cell(x_t, h_t)
                output = self.sequence_projection(h_t)
                prob = F.softmax(output, dim=-1)
                next_aa = torch.multinomial(prob, 1).squeeze(-1)
                generated_sequences[:, t] = next_aa
                x_t = self.node_embedding(
                    F.one_hot(next_aa, num_classes=self.amino_acid_vocab_size).float()
                )

            return {
                'generated_sequence': generated_sequences,
                'adjacency_matrix': predicted_adjacency
            }

    def compute_loss(self, predictions, targets):
        """
        Compute combined loss for sequence and adjacency matrix predictions

        Args:
            predictions: Dictionary from forward pass
            targets: Dictionary with 'sequence' and 'adjacency_matrix'

        Returns:
            Combined loss
        """
        # Sequence loss (cross entropy)
        seq_logits = predictions['sequence_logits']
        target_seq = targets['sequence']

        batch_size, seq_len, vocab_size = seq_logits.size()
        seq_logits_flat = seq_logits.view(-1, vocab_size)
        target_seq_flat = target_seq.view(-1)

        sequence_loss = F.cross_entropy(seq_logits_flat, target_seq_flat)

        # Adjacency matrix loss (binary cross entropy)
        pred_adj = predictions['adjacency_matrix']
        target_adj = targets['adjacency_matrix']

        # Apply mask to only consider non-diagonal elements
        mask = 1 - torch.eye(pred_adj.size(1), device=pred_adj.device).unsqueeze(0)
        adjacency_loss = F.binary_cross_entropy(
            pred_adj * mask,
            target_adj * mask,
            reduction='sum'
        ) / (mask.sum() + 1e-8)  # Normalize by number of edges

        # Combined loss (can be weighted if needed)
        combined_loss = sequence_loss + adjacency_loss

        return {
            'combined_loss': combined_loss,
            'sequence_loss': sequence_loss,
            'adjacency_loss': adjacency_loss
        }


# Training function with dual outputs
def train_dual_output_model(model, train_loader, val_loader, num_epochs=10, lr=1e-4, device='cpu'):
    """Train the dual output GRAN model"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses = {'combined': [], 'sequence': [], 'adjacency': []}
    val_losses = {'combined': [], 'sequence': [], 'adjacency': []}

    for epoch in range(num_epochs):
        # Training
        model.train()
        epoch_train_losses = {'combined': 0, 'sequence': 0, 'adjacency': 0}
        batch_count = 0

        for i, (aa_seqs, adjacency_matrices, node_feats) in enumerate(train_loader):
            # Skip empty batches
            if len(aa_seqs) == 0:
                continue

            # Move data to device
            aa_seqs = aa_seqs.to(device)
            adjacency_matrices = adjacency_matrices.to(device)
            node_feats = node_feats.to(device)

            try:
                # Forward pass
                predictions = model(
                    node_feats,
                    adjacency_matrices,
                    target_sequences=aa_seqs,
                    target_adjacency=adjacency_matrices
                )

                # Compute loss
                targets = {
                    'sequence': aa_seqs,
                    'adjacency_matrix': adjacency_matrices
                }

                losses = model.compute_loss(predictions, targets)

                # Backward pass and optimization
                optimizer.zero_grad()
                losses['combined_loss'].backward()
                optimizer.step()

                # Record losses
                epoch_train_losses['combined'] += losses['combined_loss'].item()
                epoch_train_losses['sequence'] += losses['sequence_loss'].item()
                epoch_train_losses['adjacency'] += losses['adjacency_loss'].item()
                batch_count += 1

                if i % 10 == 0:
                    print(f"Epoch {epoch+1}, Batch {i}: Combined Loss: {losses['combined_loss'].item():.4f}, "
                          f"Seq Loss: {losses['sequence_loss'].item():.4f}, "
                          f"Adj Loss: {losses['adjacency_loss'].item():.4f}")

            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        # Average train losses
        for key in epoch_train_losses:
            if batch_count > 0:
                epoch_train_losses[key] /= batch_count
                train_losses[key].append(epoch_train_losses[key])

        # Validation
        model.eval()
        epoch_val_losses = {'combined': 0, 'sequence': 0, 'adjacency': 0}
        val_batch_count = 0

        with torch.no_grad():
            for aa_seqs, adjacency_matrices, node_feats in val_loader:
                if len(aa_seqs) == 0:
                    continue

                aa_seqs = aa_seqs.to(device)
                adjacency_matrices = adjacency_matrices.to(device)
                node_feats = node_feats.to(device)

                try:
                    # Forward pass
                    predictions = model(
                        node_feats,
                        adjacency_matrices,
                        target_sequences=aa_seqs,
                        target_adjacency=adjacency_matrices
                    )

                    # Compute loss
                    targets = {
                        'sequence': aa_seqs,
                        'adjacency_matrix': adjacency_matrices
                    }

                    losses = model.compute_loss(predictions, targets)

                    # Record losses
                    epoch_val_losses['combined'] += losses['combined_loss'].item()
                    epoch_val_losses['sequence'] += losses['sequence_loss'].item()
                    epoch_val_losses['adjacency'] += losses['adjacency_loss'].item()
                    val_batch_count += 1

                except Exception as e:
                    print(f"Error in validation: {e}")
                    continue

        # Average validation losses
        for key in epoch_val_losses:
            if val_batch_count > 0:
                epoch_val_losses[key] /= val_batch_count
                val_losses[key].append(epoch_val_losses[key])

        # Print epoch summary
        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"Train: Combined {epoch_train_losses['combined']:.4f}, "
              f"Seq {epoch_train_losses['sequence']:.4f}, "
              f"Adj {epoch_train_losses['adjacency']:.4f} | "
              f"Val: Combined {epoch_val_losses['combined']:.4f}, "
              f"Seq {epoch_val_losses['sequence']:.4f}, "
              f"Adj {epoch_val_losses['adjacency']:.4f}")

    return train_losses, val_losses


# Inference function
def generate_protein(model, adjacency_matrix, node_features, device, max_length=100, unique_aa=None):
    """
    Generate both a protein sequence and adjacency matrix using the dual output GRAN model

    Args:
        model: Trained DualOutputGRAN model
        adjacency_matrix: Input adjacency matrix
        node_features: Input node features
        device: Device to run inference on
        max_length: Maximum sequence length to generate
        unique_aa: List of amino acids for decoding

    Returns:
        Dictionary with generated sequence and adjacency matrix
    """
    # Prepare inputs
    if len(adjacency_matrix.shape) == 2:
        adjacency_matrix = adjacency_matrix.unsqueeze(0)
    if len(node_features.shape) == 2:
        node_features = node_features.unsqueeze(0)

    adjacency_matrix = adjacency_matrix.to(device)
    node_features = node_features.to(device)

    # Inference
    model.eval()
    with torch.no_grad():
        outputs = model(node_features, adjacency_matrix, max_length=max_length)

    # Decode sequence
    generated_ids = outputs['generated_sequence'][0]

    if unique_aa is None:
        amino_acids = "ACDEFGHIKLMNPQRSTVWYX"
    else:
        amino_acids = unique_aa

    protein_sequence = ""
    for aa_id in generated_ids:
        if aa_id.item() < len(amino_acids):
            protein_sequence += amino_acids[aa_id.item()]

    # Get predicted adjacency matrix
    predicted_adjacency = outputs['adjacency_matrix'][0].cpu().numpy()

    return {
        'protein_sequence': protein_sequence,
        'adjacency_matrix': predicted_adjacency
    }

In [ ]:
import pickle as pkl
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import os
import random
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
import networkx as nx


class GraphAttention(nn.Module):
    def __init__(self, in_features, out_features, n_heads=4, dropout=0.1, alpha=0.2):
        super(GraphAttention, self).__init__()
        self.in_features = in_features
        self.out_features = out_features  # out_features_per_head
        self.n_heads = n_heads
        self.dropout = dropout
        self.alpha = alpha

        # Linear transformations
        self.W = nn.Linear(in_features, n_heads * out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)

        # Initialize parameters
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a.weight)

        self.leakyrelu = nn.LeakyReLU(self.alpha)
        self.dropout_layer = nn.Dropout(self.dropout)

    def forward(self, h, adj):
        # h: [batch_size, num_nodes, in_features] (32, 50, 128)
        # adj: [batch_size, num_nodes, num_nodes] (32, 50, 50)

        batch_size, num_nodes = h.size(0), h.size(1)

        # Linear transformation
        Wh = self.W(h)  # [32, 50, n_heads*out_features] (32,50,4*32=128)
        Wh = Wh.view(batch_size, num_nodes, self.n_heads, self.out_features)  # [32,50,4,32]
        Wh = Wh.permute(0, 2, 1, 3)  # [32,4,50,32]

        # Compute attention coefficients
        Wh_repeated_i = Wh.unsqueeze(3).expand(-1, -1, -1, num_nodes, -1)  # [32,4,50,50,32]
        Wh_repeated_j = Wh.unsqueeze(2).expand(-1, -1, num_nodes, -1, -1)  # [32,4,50,50,32]
        concat = torch.cat([Wh_repeated_i, Wh_repeated_j], dim=-1)  # [32,4,50,50,64]

        e = self.leakyrelu(self.a(concat).squeeze(-1))  # [32,4,50,50]

        # Mask attention coefficients
        zero_vec = -9e15 * torch.ones_like(e)
        adj = adj.unsqueeze(1)  # [32,1,50,50]
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=-1)  # [32,4,50,50]
        attention = self.dropout_layer(attention)

        # Apply attention
        h_prime = torch.matmul(attention, Wh)  # [32,4,50,32]
        h_prime = h_prime.permute(0, 2, 1, 3).contiguous()  # [32,50,4,32]
        h_prime = h_prime.view(batch_size, num_nodes, -1)  # [32,50,128]

        return h_prime


class DualOutputGRAN(nn.Module):
    """
    Graph Recurrent Attention Network that generates both adjacency matrices and amino acid sequences.
    """
    def __init__(self, node_features=22, hidden_dim=128, num_layers=2,
                 n_heads=4, dropout=0.1, amino_acid_vocab_size=22):
        super(DualOutputGRAN, self).__init__()
        self.node_features = node_features
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.n_heads = n_heads
        self.amino_acid_vocab_size = amino_acid_vocab_size

        assert hidden_dim % n_heads == 0, "hidden_dim must be divisible by n_heads"
        self.out_features_per_head = hidden_dim // n_heads

        # Node feature embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # Graph attention layers
        self.gat_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.gat_layers.append(GraphAttention(
                in_features=hidden_dim,
                out_features=self.out_features_per_head,
                n_heads=n_heads,
                dropout=dropout
            ))

        # Sequence generation
        self.rnn_cell = nn.GRUCell(hidden_dim, hidden_dim)
        self.sequence_projection = nn.Linear(hidden_dim, amino_acid_vocab_size)

        # Adjacency matrix generation
        self.edge_predictor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

        self.dropout = nn.Dropout(dropout)

    def _process_graph(self, node_features, adjacency_matrix):
        """Process the input graph with graph attention layers"""
        h = self.node_embedding(node_features)  # [32,50,22] -> [32,50,128]

        for gat_layer in self.gat_layers:
            h = gat_layer(h, adjacency_matrix)  # [32,50,128]
            h = F.elu(h)
            h = self.dropout(h)

        return h

    def _generate_adjacency(self, node_embeddings):
        """Generate adjacency matrix from node embeddings"""
        batch_size, num_nodes, _ = node_embeddings.size()

        # Create all pairwise combinations of node embeddings
        node_i = node_embeddings.unsqueeze(2).repeat(1, 1, num_nodes, 1)  # [B, N, N, H]
        node_j = node_embeddings.unsqueeze(1).repeat(1, num_nodes, 1, 1)  # [B, N, N, H]

        # Concatenate node pairs
        node_pairs = torch.cat([node_i, node_j], dim=-1)  # [B, N, N, 2H]

        # Reshape for passing through edge predictor
        flat_pairs = node_pairs.view(-1, 2 * self.hidden_dim)  # [B*N*N, 2H]

        # Predict edges
        edge_scores = self.edge_predictor(flat_pairs).view(batch_size, num_nodes, num_nodes)  # [B, N, N]

        # Ensure symmetry (for undirected graphs)
        edge_scores = (edge_scores + edge_scores.transpose(1, 2)) / 2

        return edge_scores

    def forward(self, node_features, adjacency_matrix, target_sequences=None, target_adjacency=None, max_length=100):
        """
        Forward pass with dual outputs (sequence and adjacency matrix)

        Args:
            node_features: [batch_size, num_nodes, node_feature_dim]
            adjacency_matrix: [batch_size, num_nodes, num_nodes]
            target_sequences: [batch_size, seq_length] (for training)
            target_adjacency: [batch_size, num_nodes, num_nodes] (for training)
            max_length: Maximum sequence length for generation

        Returns:
            Dictionary containing:
                - 'sequence_logits': Predicted sequence logits
                - 'adjacency_matrix': Predicted adjacency matrix
                - Or generated sequence and adjacency matrix during inference
        """
        batch_size, num_nodes = node_features.size(0), node_features.size(1)

        # Process graph
        node_embeddings = self._process_graph(node_features, adjacency_matrix)  # [B, N, H]

        # Generate adjacency matrix
        predicted_adjacency = self._generate_adjacency(node_embeddings)  # [B, N, N]

        # Initialize RNN for sequence generation
        graph_embedding = torch.mean(node_embeddings, dim=1)  # [B, H]
        h_t = graph_embedding  # Initial hidden state

        if target_sequences is not None:
            # Training mode
            seq_length = target_sequences.size(1)
            sequence_logits = torch.zeros(batch_size, seq_length, self.amino_acid_vocab_size,
                                          device=node_features.device)

            x_t = torch.zeros(batch_size, self.hidden_dim, device=node_features.device)

            for t in range(seq_length):
                h_t = self.rnn_cell(x_t, h_t)  # [B, H]
                sequence_logits[:, t, :] = self.sequence_projection(h_t)  # [B, vocab_size]

                if t < seq_length - 1:
                    x_t = self.node_embedding(
                        F.one_hot(target_sequences[:, t],
                                  num_classes=self.amino_acid_vocab_size).float()
                    )  # [B, H]

            return {
                'sequence_logits': sequence_logits,
                'adjacency_matrix': predicted_adjacency
            }

        else:
            # Generation mode
            generated_sequences = torch.zeros(batch_size, max_length,
                                              dtype=torch.long,
                                              device=node_features.device)
            x_t = torch.zeros(batch_size, self.hidden_dim,
                              device=node_features.device)

            for t in range(max_length):
                h_t = self.rnn_cell(x_t, h_t)
                output = self.sequence_projection(h_t)
                prob = F.softmax(output, dim=-1)
                next_aa = torch.multinomial(prob, 1).squeeze(-1)
                generated_sequences[:, t] = next_aa
                x_t = self.node_embedding(
                    F.one_hot(next_aa, num_classes=self.amino_acid_vocab_size).float()
                )

            return {
                'generated_sequence': generated_sequences,
                'adjacency_matrix': predicted_adjacency
            }

    def compute_loss(self, predictions, targets):
        """
        Compute combined loss for sequence and adjacency matrix predictions

        Args:
            predictions: Dictionary from forward pass
            targets: Dictionary with 'sequence' and 'adjacency_matrix'

        Returns:
            Combined loss
        """
        # Sequence loss (cross entropy)
        seq_logits = predictions['sequence_logits']
        target_seq = targets['sequence']

        batch_size, seq_len, vocab_size = seq_logits.size()
        seq_logits_flat = seq_logits.view(-1, vocab_size)
        target_seq_flat = target_seq.view(-1)

        sequence_loss = F.cross_entropy(seq_logits_flat, target_seq_flat)

        # Adjacency matrix loss (binary cross entropy)
        pred_adj = predictions['adjacency_matrix']
        target_adj = targets['adjacency_matrix']

        # Apply mask to only consider non-diagonal elements
        mask = 1 - torch.eye(pred_adj.size(1), device=pred_adj.device).unsqueeze(0)
        adjacency_loss = F.binary_cross_entropy(
            pred_adj * mask,
            target_adj * mask,
            reduction='sum'
        ) / (mask.sum() + 1e-8)  # Normalize by number of edges

        # Combined loss (can be weighted if needed)
        combined_loss = sequence_loss + adjacency_loss

        return {
            'combined_loss': combined_loss,
            'sequence_loss': sequence_loss,
            'adjacency_loss': adjacency_loss
        }


def load_protein_graph_data(parent_folder, max_proteins=None, chunk_length=50):
    """
    Load protein NetworkX graph data directly from *_graph.pkl files
    """
    random.seed(42)

    # List to store graphs and their associated amino acid sequences
    protein_graphs = []
    protein_sequences = []

    folders = [name for name in os.listdir(parent_folder)
               if os.path.isdir(os.path.join(parent_folder, name))]

    print(f"Found {len(folders)} protein folders")

    # Load only max_proteins if specified
    if max_proteins:
        folders = folders[:max_proteins]

    for folder in folders:
        # Look for graph files in the folder
        folder_path = os.path.join(parent_folder, folder)
        graph_files = [f for f in os.listdir(folder_path) if f.endswith('_graph.pkl')]

        if graph_files:
            graph_file = os.path.join(folder_path, graph_files[0])  # Take the first graph file
            try:
                with open(graph_file, 'rb') as f:
                    graph = pkl.load(f)

                    # Extract amino acid sequence from the graph
                    # The actual AA info is in the 'residue_name' attribute
                    aa_seq = []
                    # Sort nodes to maintain the correct order
                    sorted_nodes = sorted(graph.nodes(), key=lambda x:
                    int(graph.nodes[x]['residue_number'])
                    if 'residue_number' in graph.nodes[x] else 0)

                    for node in sorted_nodes:
                        if 'residue_name' in graph.nodes[node]:
                            aa_seq.append(graph.nodes[node]['residue_name'])
                        else:
                            # If amino acid info is not available, use a placeholder
                            aa_seq.append('X')

                    protein_graphs.append(graph)
                    protein_sequences.append(aa_seq)
                    print(f"Loaded graph from {graph_file} with {len(aa_seq)} amino acids")
            except Exception as e:
                print(f"Error loading {graph_file}: {e}")

    if not protein_graphs:
        raise ValueError("No protein graph files found! Check the file pattern and paths.")

    print(f"Loaded {len(protein_graphs)} protein graphs")

    # Create smaller subgraphs if needed (this would require graph manipulation)
    # This is a simplified approach - ideally you'd want to extract connected subgraphs
    subgraphs = []
    subsequences = []

    if chunk_length < max([len(seq) for seq in protein_sequences]):
        for i, (graph, sequence) in enumerate(zip(protein_graphs, protein_sequences)):
            # Extract nodes by residue number ranges
            sorted_nodes = sorted(graph.nodes(), key=lambda x:
            int(graph.nodes[x]['residue_number'])
            if 'residue_number' in graph.nodes[x] else 0)

            for start_idx in range(0, max(1, len(sorted_nodes) - chunk_length + 1), chunk_length // 2):
                end_idx = min(start_idx + chunk_length, len(sorted_nodes))
                node_subset = sorted_nodes[start_idx:end_idx]

                if node_subset:
                    subgraph = graph.subgraph(node_subset)
                    if len(subgraph) > 0:
                        subgraphs.append(subgraph)
                        subsequences.append(sequence[start_idx:end_idx])
    else:
        subgraphs = protein_graphs
        subsequences = protein_sequences

    # If no subgraphs were created, use the full graphs
    if not subgraphs:
        print("Warning: Could not create subgraphs. Using full graphs instead.")
        subgraphs = protein_graphs
        subsequences = protein_sequences

    # Shuffle the data
    combined = list(zip(subgraphs, subsequences))
    random.shuffle(combined)
    subgraphs, subsequences = zip(*combined)

    # Limit size if needed
    if max_proteins and len(subgraphs) > max_proteins * 10:
        subgraphs = subgraphs[:max_proteins * 10]
        subsequences = subsequences[:max_proteins * 10]

    print(f"Created {len(subgraphs)} protein subgraphs for training")

    return protein_graphs, protein_sequences, list(subgraphs), list(subsequences)


def get_config_ranges(all_distances, n_ranges=6, mode='percentile'):
    """
    Define contact map configuration ranges
    """
    if mode == 'percentile':
        # even distribution of contacts within the ranges
        limits = np.percentile(all_distances, [100*i/n_ranges for i in range(n_ranges)] + [99.6]).tolist()
    elif mode == 'distance':
        # even distance ranges
        min_d, max_d = all_distances.min(), all_distances.max()
        std, mean = all_distances.std(), all_distances.mean()
        min_std, max_std = mean-2*std, mean+2*std
        limits = [min_std + i*((max_std-min_std)/n_ranges) for i in range(n_ranges)] + [max_d]
        limits[0] = min_d
    else:
        print("Define config mode, mode in ['percentile', 'distances']")
        return False

    print('Limits', limits)

    CONFIGS = []
    for i in range(len(limits)-1):
        CONFIGS.append({'lower': limits[i], 'upper': limits[i+1]})

    return CONFIGS


def prepare_graph_data_for_training(subgraphs, subsequences, unique_aa):
    """
    Prepare graph data for GRAN model training

    Args:
        subgraphs: List of NetworkX subgraphs
        subsequences: List of amino acid sequences corresponding to the subgraphs
        unique_aa: Set of unique amino acids to use for one-hot encoding

    Returns:
        aa_sequences_tensor: Tensor of amino acid sequences (targets)
        adjacency_tensors: Tensor of graph adjacency matrices
        node_features_tensor: Tensor of node features
    """
    # Prepare amino acid sequences (these will be our targets)
    aa_sequences = []
    for seq in subsequences:
        # Convert amino acid sequence to indices
        aa_indices = []
        for aa in seq:
            if aa in unique_aa:
                aa_indices.append(list(unique_aa).index(aa))
            else:
                # Handle unknown amino acids
                aa_indices.append(list(unique_aa).index('X') if 'X' in unique_aa else 0)
        aa_sequences.append(aa_indices)

    # Prepare adjacency matrices and node features
    adjacency_matrices = []
    node_features = []

    for i, graph in enumerate(subgraphs):
        # Get number of nodes
        num_nodes = len(graph)

        # Skip empty graphs
        if num_nodes == 0:
            continue

        # Create adjacency matrix
        adj_matrix = nx.to_numpy_array(graph)
        adjacency_matrices.append(adj_matrix)

        # Extract ordered nodes
        sorted_nodes = sorted(graph.nodes(), key=lambda x:
        int(graph.nodes[x]['residue_number'])
        if 'residue_number' in graph.nodes[x] else 0)

        # Create node features - use Meiler embedding if available
        features = np.zeros((num_nodes, len(unique_aa)))

        for j, node in enumerate(sorted_nodes):
            # Try to use meiler features if available (more informative)
            if 'meiler' in graph.nodes[node]:
                # Replace features with a better representation
                # For now, use one-hot as fallback
                aa = graph.nodes[node]['residue_name']
                if aa in unique_aa:
                    features[j, list(unique_aa).index(aa)] = 1.0
                elif 'X' in unique_aa:
                    features[j, list(unique_aa).index('X')] = 1.0
                else:
                    features[j, 0] = 1.0
            # Fallback to one-hot encoding
            elif 'residue_name' in graph.nodes[node]:
                aa = graph.nodes[node]['residue_name']
                if aa in unique_aa:
                    features[j, list(unique_aa).index(aa)] = 1.0
                elif 'X' in unique_aa:
                    features[j, list(unique_aa).index('X')] = 1.0
                else:
                    features[j, 0] = 1.0

        node_features.append(features)

    # Check if we have data
    if not adjacency_matrices or not node_features:
        raise ValueError("No valid graphs found after processing")

    # Convert to tensors
    aa_sequences_tensor = [torch.tensor(seq, dtype=torch.long) for seq in aa_sequences if seq]
    adjacency_tensors = [torch.tensor(adj, dtype=torch.float32) for adj in adjacency_matrices]
    node_features_tensor = [torch.tensor(nf, dtype=torch.float32) for nf in node_features]

    return aa_sequences_tensor, adjacency_tensors, node_features_tensor


def collate_batch(batch):
    """
    Custom collate function for padding sequences of variable length
    """
    # Handle potentially empty batches
    if not batch:
        return [], [], []

    aa_seqs, adjacency_matrices, node_feats = zip(*batch)

    # Handle potentially empty elements
    if not aa_seqs or not adjacency_matrices or not node_feats:
        return [], [], []

    # Pad sequences
    padded_aa_seqs = torch.nn.utils.rnn.pad_sequence(aa_seqs, batch_first=True)

    # Get maximum dimensions
    max_len = max([adj.size(0) for adj in adjacency_matrices])

    padded_adjacency_matrices = []
    padded_node_feats = []

    for i in range(len(adjacency_matrices)):
        adj = adjacency_matrices[i]
        nf = node_feats[i]

        pad_size = max_len - adj.size(0)
        if pad_size > 0:
            # Pad adjacency matrix
            padded_adj = F.pad(adj, (0, pad_size, 0, pad_size), "constant", 0)
            padded_adjacency_matrices.append(padded_adj)

            # Pad node features - make sure pad size isn't larger than twice the feature dim
            safe_pad_size = min(pad_size, nf.size(1) * 2)
            if safe_pad_size < pad_size:
                # We need to truncate the padding
                padded_nf = F.pad(nf, (0, 0, 0, safe_pad_size), "constant", 0)
                # Then add more nodes with zero features to match the adjacency matrix
                additional_pad = pad_size - safe_pad_size
                zeros = torch.zeros(additional_pad, nf.size(1))
                padded_nf = torch.cat([padded_nf, zeros], dim=0)
            else:
                padded_nf = F.pad(nf, (0, 0, 0, pad_size), "constant", 0)

            padded_node_feats.append(padded_nf)
        else:
            padded_adjacency_matrices.append(adj)
            padded_node_feats.append(nf)

    # Stack the padded tensors
    padded_adjacency_matrices = torch.stack(padded_adjacency_matrices)
    padded_node_feats = torch.stack(padded_node_feats)

    return padded_aa_seqs, padded_adjacency_matrices, padded_node_feats


def create_dataloader(aa_sequences, adjacency_matrices, node_features, batch_size=32, shuffle=True):
    """
    Create a dataloader from the prepared data
    """
    dataset = list(zip(aa_sequences, adjacency_matrices, node_features))
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_batch
    )
    return dataloader


def train_dual_output_model(model, train_loader, val_loader, num_epochs=10, lr=1e-4, device='cpu'):
    """Train the dual output GRAN model"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses = {'combined': [], 'sequence': [], 'adjacency': []}
    val_losses = {'combined': [], 'sequence': [], 'adjacency': []}

    for epoch in range(num_epochs):
        # Training
        model.train()
        epoch_train_losses = {'combined': 0, 'sequence': 0, 'adjacency': 0}
        batch_count = 0

        for i, (aa_seqs, adjacency_matrices, node_feats) in enumerate(train_loader):
            # Skip empty batches
            if len(aa_seqs) == 0:
                continue

            # Move data to device
            aa_seqs = aa_seqs.to(device)
            adjacency_matrices = adjacency_matrices.to(device)
            node_feats = node_feats.to(device)

            try:
                # Forward pass
                predictions = model(
                    node_feats,
                    adjacency_matrices,
                    target_sequences=aa_seqs,
                    target_adjacency=adjacency_matrices
                )

                # Compute loss
                targets = {
                    'sequence': aa_seqs,
                    'adjacency_matrix': adjacency_matrices
                }

                losses = model.compute_loss(predictions, targets)

                # Backward pass and optimization
                optimizer.zero_grad()
                losses['combined_loss'].backward()
                optimizer.step()

                # Record losses
                epoch_train_losses['combined'] += losses['combined_loss'].item()
                epoch_train_losses['sequence'] += losses['sequence_loss'].item()
                epoch_train_losses['adjacency'] += losses['adjacency_loss'].item()
                batch_count += 1

                if i % 10 == 0:
                    print(f"Epoch {epoch+1}, Batch {i}: Combined Loss: {losses['combined_loss'].item():.4f}, "
                          f"Seq Loss: {losses['sequence_loss'].item():.4f}, "
                          f"Adj Loss: {losses['adjacency_loss'].item():.4f}")

            except Exception as e:
                print(f"Error in batch {i}: {e}")
                continue

        # Average train losses
        for key in epoch_train_losses:
            if batch_count > 0:
                epoch_train_losses[key] /= batch_count
                train_losses[key].append(epoch_train_losses[key])

        # Validation
        model.eval()
        epoch_val_losses = {'combined': 0, 'sequence': 0, 'adjacency': 0}
        val_batch_count = 0

        with torch.no_grad():
            for aa_seqs, adjacency_matrices, node_feats in val_loader:
                if len(aa_seqs) == 0:
                    continue

                aa_seqs = aa_seqs.to(device)
                adjacency_matrices = adjacency_matrices.to(device)
                node_feats = node_feats.to(device)

                try:
                    # Forward pass
                    predictions = model(
                        node_feats,
                        adjacency_matrices,
                        target_sequences=aa_seqs,
                        target_adjacency=adjacency_matrices
                    )

                    # Compute loss
                    targets = {
                        'sequence': aa_seqs,
                        'adjacency_matrix': adjacency_matrices
                    }

                    losses = model.compute_loss(predictions, targets)

                    # Record losses
                    epoch_val_losses['combined'] += losses['combined_loss'].item()
                    epoch_val_losses['sequence'] += losses['sequence_loss'].item()
                    epoch_val_losses['adjacency'] += losses['adjacency_loss'].item()
                    val_batch_count += 1

                except Exception as e:
                    print(f"Error in validation: {e}")
                    continue

        # Average validation losses
        for key in epoch_val_losses:
            if val_batch_count > 0:
                epoch_val_losses[key] /= val_batch_count
                val_losses[key].append(epoch_val_losses[key])

        # Print epoch summary
        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"Train: Combined {epoch_train_losses['combined']:.4f}, "
              f"Seq {epoch_train_losses['sequence']:.4f}, "
              f"Adj {epoch_train_losses['adjacency']:.4f} | "
              f"Val: Combined {epoch_val_losses['combined']:.4f}, "
              f"Seq {epoch_val_losses['sequence']:.4f}, "
              f"Adj {epoch_val_losses['adjacency']:.4f}")

    return train_losses, val_losses


def generate_protein(model, adjacency_matrix, node_features, device, max_length=100, unique_aa=None):
    """
    Generate both a protein sequence and adjacency matrix using the dual output GRAN model

    Args:
        model: Trained DualOutputGRAN model
        adjacency_matrix: Input adjacency matrix
        node_features: Input node features
        device: Device to run inference on
        max_length: Maximum sequence length to generate
        unique_aa: List of amino acids for decoding

    Returns:
        Dictionary with generated sequence and adjacency matrix
    """
    # Prepare inputs
    if len(adjacency_matrix.shape) == 2:
        adjacency_matrix = adjacency_matrix.unsqueeze(0)
    if len(node_features.shape) == 2:
        node_features = node_features.unsqueeze(0)

    adjacency_matrix = adjacency_matrix.to(device)
    node_features = node_features.to(device)

    # Inference
    model.eval()
    with torch.no_grad():
        outputs = model(node_features, adjacency_matrix, max_length=max_length)

    # Decode sequence
    generated_ids = outputs['generated_sequence'][0]

    if unique_aa is None:
        amino_acids = "ACDEFGHIKLMNPQRSTVWYX"
    else:
        amino_acids = unique_aa

    protein_sequence = ""
    for aa_id in generated_ids:
        if aa_id.item() < len(amino_acids):
            protein_sequence += amino_acids[aa_id.item()]

    # Get predicted adjacency matrix
    predicted_adjacency = outputs['adjacency_matrix'][0].cpu().numpy()

    return {
        'protein_sequence': protein_sequence,
        'adjacency_matrix': predicted_adjacency
    }




In [ ]:
def main():
    # Parameters
    parent_folder = "nanos_networkx_small"  # Update this to your data path
    chunk_length = 50
    max_proteins = 2000  # Limit number of proteins for faster execution

    # Set device
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load protein graph data directly
    try:
        full_graphs, full_sequences, subgraphs, subsequences = load_protein_graph_data(
            parent_folder, max_proteins, chunk_length
        )
    except Exception as e:
        print(f"Error loading graph data: {e}")
        print("Current directory contains:", os.listdir())
        return

    # First graph debug
    print("First few nodes of first graph:")
    first_graph = full_graphs[0]
    for i, node in enumerate(sorted(first_graph.nodes())[:5]):
        print(f"Node {node} attributes: {first_graph.nodes[node]}")

    # Define a standard set of amino acids (all 20 standard ones)
    STANDARD_AA = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
                   'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'X']

    # Get unique amino acids from sequences but ensure we have at least the standard 20
    UNIQUE_AA = set()
    for seq in full_sequences:
        UNIQUE_AA.update(seq)

    # Merge with standard AAs
    UNIQUE_AA = sorted(list(set(UNIQUE_AA).union(set(STANDARD_AA))))
    print(f"Unique amino acids: {len(UNIQUE_AA)}")
    print(f"Amino acids found: {', '.join(UNIQUE_AA)}")

    # Prepare data for training using graph data
    aa_sequences, adjacency_matrices, node_features = prepare_graph_data_for_training(
        subgraphs, subsequences, UNIQUE_AA
    )

    print(f"Prepared data: {len(aa_sequences)} sequences, {len(adjacency_matrices)} adjacency matrices")

    # Split into train and validation sets
    split_idx = int(0.8 * len(aa_sequences))
    train_aa = aa_sequences[:split_idx]
    train_adj = adjacency_matrices[:split_idx]
    train_nf = node_features[:split_idx]

    val_aa = aa_sequences[split_idx:]
    val_adj = adjacency_matrices[split_idx:]
    val_nf = node_features[split_idx:]

    # Create dataloaders
    train_loader = create_dataloader(train_aa, train_adj, train_nf, batch_size=32)
    val_loader = create_dataloader(val_aa, val_adj, val_nf, batch_size=32)

    # Model parameters
    node_features_dim = node_features[0].size(1)  # Feature dimension
    hidden_dim = max(128, node_features_dim * 2)  # Ensure hidden dim is at least 2x input
    num_layers = 2
    n_heads = 4
    amino_acid_vocab_size = len(UNIQUE_AA)

    print(f"Model configuration:")
    print(f"- Node features dimension: {node_features_dim}")
    print(f"- Hidden dimension: {hidden_dim}")
    print(f"- Number of graph layers: {num_layers}")
    print(f"- Number of attention heads: {n_heads}")
    print(f"- Amino acid vocabulary size: {amino_acid_vocab_size}")

    # Create the dual output model
    model = DualOutputGRAN(
        node_features=node_features_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        n_heads=n_heads,
        dropout=0.1,
        amino_acid_vocab_size=amino_acid_vocab_size
    ).to(device)

    # Sample testing to verify forward pass works
    print("Testing model with sample data...")
    with torch.no_grad():
        # Create small test batch
        test_node_feats = train_nf[0].unsqueeze(0).to(device)
        test_adj = train_adj[0].unsqueeze(0).to(device)
        test_seq = train_aa[0].unsqueeze(0).to(device)

        # Forward pass
        outputs = model(test_node_feats, test_adj, test_seq)
        print("Forward pass successful!")
        print(f"Sequence logits shape: {outputs['sequence_logits'].shape}")
        print(f"Adjacency matrix shape: {outputs['adjacency_matrix'].shape}")

    # Train model using the dual output training function
    print("\nStarting model training...")
    train_losses, val_losses = train_dual_output_model(
        model, train_loader, val_loader,
        num_epochs=250, lr=1e-4, device=device
    )

    # Plot training progress (expanded to show both loss components)
    plt.figure(figsize=(15, 5))

    # Sequence loss plot
    plt.subplot(1, 3, 1)
    plt.plot(train_losses['sequence'], label='Train Loss')
    plt.plot(val_losses['sequence'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Sequence Loss')
    plt.title('Sequence Generation')
    plt.legend()

    # Adjacency loss plot
    plt.subplot(1, 3, 2)
    plt.plot(train_losses['adjacency'], label='Train Loss')
    plt.plot(val_losses['adjacency'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Adjacency Loss')
    plt.title('Adjacency Prediction')
    plt.legend()

    # Combined loss plot
    plt.subplot(1, 3, 3)
    plt.plot(train_losses['combined'], label='Train Loss')
    plt.plot(val_losses['combined'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Combined Loss')
    plt.title('Combined Performance')
    plt.legend()

    plt.tight_layout()
    plt.show()

    # Generate a sample protein (both sequence and structure)
    if len(subgraphs) > 0:
        print("\nGenerating sample protein...")
        sample_graph = subgraphs[0]
        sample_seq = subsequences[0]

        # Convert to tensors for generation
        sample_adj = torch.tensor(adjacency_matrices[0], dtype=torch.float32)
        sample_features = torch.tensor(node_features[0], dtype=torch.float32)

        # Generate protein with dual outputs
        results = generate_protein(
            model, sample_adj, sample_features, device,
            max_length=len(sample_seq),
            unique_aa=UNIQUE_AA
        )

        print("\nOriginal sequence:", ''.join(sample_seq[:20]) + "..." if len(sample_seq) > 20 else ''.join(sample_seq))
        print("Generated sequence:", results['protein_sequence'][:20] + "..." if len(results['protein_sequence']) > 20 else results['protein_sequence'])
        print("Generated adjacency matrix shape:", results['adjacency_matrix'].shape)

        # Visualize original vs generated adjacency matrices
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(adjacency_matrices[0].numpy(), cmap='viridis')
        plt.title("Original Adjacency Matrix")
        plt.colorbar()

        plt.subplot(1, 2, 2)
        plt.imshow(results['adjacency_matrix'], cmap='viridis')
        plt.title("Generated Adjacency Matrix")
        plt.colorbar()

        plt.tight_layout()
        plt.show()

        # Calculate accuracy metrics
        seq_match_percent = sum([1 if a == b else 0 for a, b in
                                 zip(sample_seq[:len(results['protein_sequence'])],
                                     results['protein_sequence'])]) / len(results['protein_sequence']) * 100

        # Convert adjacency matrices to binary for comparison
        orig_adj_binary = (adjacency_matrices[0].numpy() > 0.5).astype(float)
        pred_adj_binary = (results['adjacency_matrix'] > 0.5).astype(float)

        # Mask diagonal elements
        mask = np.ones_like(orig_adj_binary) - np.eye(orig_adj_binary.shape[0])
        adj_match_percent = np.sum((orig_adj_binary == pred_adj_binary) * mask) / np.sum(mask) * 100

        print(f"\nSequence match accuracy: {seq_match_percent:.2f}%")
        print(f"Adjacency matrix match accuracy: {adj_match_percent:.2f}%")

    # Save model
    torch.save(model.state_dict(), "dual_output_gran_model.pt")
    print("\nModel saved to dual_output_gran_model.pt")


if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import plotly.graph_objects as go
from matplotlib import cm
import matplotlib.colors as mcolors

def reconstruct_coords_local_plural_maps_colored(contact_maps, configs, amino_acid_seq,
                                                 hard_mask=None, sharpness=10., window=25,
                                                 dim=3, lr=1e-3, max_iter=1000, prints=10):
    """
    Reconstruct 3D coordinates from multiple contact maps with different distance constraints.
    Visualize the structure with amino acid-based coloring.

    Args:
        contact_maps: List of binary contact maps
        configs: List of dictionaries containing 'lower' and 'upper' bounds for each map
        amino_acid_seq: List or string of amino acid sequence
        hard_mask: Boolean mask for valid positions
        sharpness: Parameter for contact prediction sharpness
        window: Window size for local interactions
        dim: Dimensionality of the coordinate space (default: 3D)
        lr: Learning rate for optimization
        max_iter: Maximum number of iterations
        prints: Number of progress updates

    Returns:
        Reconstructed coordinates as numpy array and Plotly figure
    """
    print('Gradient-based optimization with multiple contact maps...')

    # Define amino acid categories for coloring
    aa_categories = {
        # Hydrophobic
        'ALA': 'hydrophobic', 'VAL': 'hydrophobic', 'LEU': 'hydrophobic',
        'ILE': 'hydrophobic', 'PHE': 'hydrophobic', 'TRP': 'hydrophobic',
        'MET': 'hydrophobic', 'PRO': 'hydrophobic',
        # Polar
        'GLY': 'polar', 'SER': 'polar', 'THR': 'polar', 'CYS': 'polar',
        'TYR': 'polar', 'ASN': 'polar', 'GLN': 'polar',
        # Positively charged
        'LYS': 'positive', 'ARG': 'positive', 'HIS': 'positive',
        # Negatively charged
        'ASP': 'negative', 'GLU': 'negative',
        # Other
        'X': 'other', 'UNK': 'other'
    }

    # Color mapping
    color_dict = {
        'hydrophobic': 'blue',
        'polar': 'green',
        'positive': 'red',
        'negative': 'orange',
        'other': 'purple'
    }

    # Convert single letter to three letter if necessary
    if len(amino_acid_seq[0]) == 1:
        one_to_three = {
            'A': 'ALA', 'C': 'CYS', 'D': 'ASP', 'E': 'GLU',
            'F': 'PHE', 'G': 'GLY', 'H': 'HIS', 'I': 'ILE',
            'K': 'LYS', 'L': 'LEU', 'M': 'MET', 'N': 'ASN',
            'P': 'PRO', 'Q': 'GLN', 'R': 'ARG', 'S': 'SER',
            'T': 'THR', 'V': 'VAL', 'W': 'TRP', 'Y': 'TYR',
            'X': 'UNK'
        }
        amino_acid_seq = [one_to_three.get(aa, 'UNK') for aa in amino_acid_seq]

    # Create color list for amino acids
    aa_colors = [color_dict[aa_categories.get(aa, 'other')] for aa in amino_acid_seq]

    min_distance = 2.3  # Minimum allowed distance between points

    N = contact_maps[0].shape[0]
    # Ensure all contact maps have the same shape
    assert len(set([cmap.shape for cmap in contact_maps])) == 1, "Contact maps must have the same dimensions"

    # Initialize coordinates as a linear chain to provide better starting point
    coords = torch.zeros((N, dim))
    for i in range(N):
        coords[i, 0] = i * 3.8  # Typical C-alpha distance
        if dim > 1:
            coords[i, 1] = 2.0 * np.sin(i * 0.5)
        if dim > 2:
            coords[i, 2] = 2.0 * np.cos(i * 0.5)

    coords.requires_grad = True

    # Use a more robust optimizer and a learning rate scheduler
    optimizer = torch.optim.Adam([coords], lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=50)

    # Convert contact maps to tensors
    tensor_maps = [torch.tensor(contact_map, dtype=torch.float32) for contact_map in contact_maps]

    # Create repulsion mask (all non-diagonal elements)
    repulsion_mask = (torch.eye(N) == 0)

    if hard_mask is None:
        # Create a mask for local interactions
        local_mask = torch.zeros_like(tensor_maps[0])
        for i in range(N):
            for j in range(max(0, i - window), min(N, i + window + 1)):
                local_mask[i, j] = 1
    else:
        local_mask = torch.tensor(hard_mask.astype(bool))

    # Calculate positive weights for each contact map - ensure they're positive
    pos_weights = []
    for i, _ in enumerate(configs):
        # Make sure pos_weight is positive by using absolute value or a minimum bound
        ratio = max(1.0, local_mask.sum() / (tensor_maps[i].sum() + 1e-8))
        pos_weight = (ratio - 1.0).detach()
        pos_weight = max(1.0, pos_weight)  # Ensure positive weight
        pos_weights.append(pos_weight)

    # Best loss tracking for early stopping
    best_loss = float('inf')
    patience_counter = 0
    best_coords = coords.clone().detach()

    # Improved BCE loss function
    bce_loss_fn = torch.nn.BCEWithLogitsLoss(reduction='none')

    for step in range(max_iter):
        optimizer.zero_grad()

        # Calculate pairwise distances
        dists = torch.cdist(coords, coords, p=2)

        # Sequential connectivity loss - ensuring connected residues are close
        sequential_dists = torch.diag(dists, diagonal=1)
        connectivity_loss = torch.mean((sequential_dists - 3.8) ** 2)  # ~3.8Å is typical C-alpha distance

        # Calculate contact predictions for each config
        total_loss = torch.tensor(0.0, requires_grad=True)
        partial_losses = []

        for i, config in enumerate(configs):
            # Calculate logits for BCE loss
            if config['lower'] == 0:
                # For upper bound only, predict contact if distance < upper
                logits = sharpness * (config['upper'] - dists)
            else:
                # For range prediction, use a composite sigmoid
                upper_logits = sharpness * (config['upper'] - dists)
                lower_logits = sharpness * (dists - config['lower'])
                # Combine logits - both conditions must be true
                logits = torch.min(upper_logits, lower_logits)

            # Apply mask to focus on meaningful interactions
            weight_matrix = torch.ones_like(tensor_maps[i])
            weight_matrix = weight_matrix * pos_weights[i] * tensor_maps[i] + (1 - tensor_maps[i])

            # Calculate loss with weights
            map_loss = bce_loss_fn(logits, tensor_maps[i])
            weighted_loss = (map_loss * weight_matrix).mean()
            partial_losses.append(weighted_loss)
            total_loss = total_loss + weighted_loss

        # Add repulsion loss to maintain minimum distances
        min_dist_violation = torch.relu(min_distance - dists)
        repulsion_loss = (min_dist_violation[repulsion_mask] ** 2).sum()

        # Add all regularization terms
        total_loss = total_loss + 0.1 * repulsion_loss + 0.05 * connectivity_loss

        # Backwards pass and optimization
        total_loss.backward()

        # Gradient clipping to prevent instability
        torch.nn.utils.clip_grad_norm_([coords], max_norm=1.0)

        optimizer.step()
        scheduler.step(total_loss)

        # Print progress and update best model if needed
        if step % max(1, int(max_iter // prints)) == 0:
            partial_losses_str = [f"{p_loss.item():8.2f}" for p_loss in partial_losses]
            print(f"Step {step:7d} | total loss: {total_loss.item():8.2f} ({', '.join(partial_losses_str)})")

            if total_loss.item() < best_loss:
                best_loss = total_loss.item()
                best_coords = coords.clone().detach()
                patience_counter = 0
            else:
                patience_counter += 1

            # Early stopping
            if patience_counter > 100:
                print("Early stopping triggered.")
                break

    # Final loss report
    partial_losses_str = [f"{p_loss.item():8.2f}" for p_loss in partial_losses]
    print(f"Step {step:7d} | total loss: {total_loss.item():8.2f} ({', '.join(partial_losses_str)}), final.")
    print('')

    # Get the coordinates
    final_coords = best_coords.detach().numpy()

    # Create visualization
    fig = visualize_protein_structure(final_coords, amino_acid_seq, aa_colors)

    return final_coords, fig

def visualize_protein_structure(coords, amino_acid_seq, colors=None):
    """
    Visualize protein structure using Plotly

    Args:
        coords: 3D coordinates of the protein structure (N x 3)
        amino_acid_seq: List of amino acid types
        colors: List of colors for each amino acid

    Returns:
        Plotly figure object
    """
    N = coords.shape[0]

    # Use rainbow colors if no colors provided
    if colors is None:
        colors = cm.rainbow(np.linspace(0, 1, N))
        colors = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b, _ in colors]

    # Ensure colors is a list of strings
    if isinstance(colors[0], str):
        color_strings = colors
    else:
        color_strings = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b, _ in colors]

    # Create figure
    fig = go.Figure()

    # Add markers for amino acids
    fig.add_trace(go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=color_strings,
            opacity=0.8
        ),
        text=[f"{i+1}: {aa}" for i, aa in enumerate(amino_acid_seq)],
        hoverinfo='text',
        name='Amino acids'
    ))

    # Add lines connecting sequential residues
    x_lines, y_lines, z_lines = [], [], []
    line_colors = []

    for i in range(N-1):
        # Add line segments connecting adjacent residues
        x_lines.extend([coords[i, 0], coords[i+1, 0], None])
        y_lines.extend([coords[i, 1], coords[i+1, 1], None])
        z_lines.extend([coords[i, 2], coords[i+1, 2], None])
        line_colors.extend([color_strings[i], color_strings[i+1], 'rgba(0,0,0,0)'])

    fig.add_trace(go.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode='lines',
        line=dict(
            color=line_colors,
            width=3
        ),
        hoverinfo='none',
        name='Backbone'
    ))

    # Configure layout
    fig.update_layout(
        title="Protein Structure",
        scene=dict(
            xaxis=dict(visible=False, showticklabels=False),
            yaxis=dict(visible=False, showticklabels=False),
            zaxis=dict(visible=False, showticklabels=False),
            bgcolor='rgba(255,255,255,1)'
        ),
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=30),
        paper_bgcolor='rgba(0,0,0,0)',
        scene_camera=dict(
            eye=dict(x=1.2, y=1.2, z=1.2)
        )
    )

    return fig

# Example usage:
# coords, fig = reconstruct_coords_local_plural_maps_colored(contact_maps, configs, amino_acid_seq)
# fig.show()

In [ ]:
# Step 1: Load the trained model and generate protein sequence and adjacency matrix
def generate_from_model(model_path, sample_adjacency, sample_features, device, unique_aa):
    """
    Generate a protein sequence and adjacency matrix from a trained model

    Args:
        model_path: Path to the saved model weights
        sample_adjacency: Sample adjacency matrix to use as input
        sample_features: Sample node features to use as input
        device: Device to run inference on (cpu or gpu)
        unique_aa: List of unique amino acids for decoding

    Returns:
        Dictionary with generated sequence and adjacency matrix
    """
    # Create model with the same architecture as during training
    node_features_dim = sample_features.size(1)
    model = DualOutputGRAN(
        node_features=node_features_dim,
        hidden_dim=128,
        num_layers=2,
        n_heads=4,
        dropout=0.1,
        amino_acid_vocab_size=len(unique_aa)
    ).to(device)

    # Load the saved weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Generate using the same function we defined earlier
    results = generate_protein(
        model, sample_adjacency, sample_features, device,
        max_length=50,  # Adjust as needed
        unique_aa=unique_aa
    )

    print("Generated protein sequence:", results['protein_sequence'])
    print("Generated adjacency matrix shape:", results['adjacency_matrix'].shape)

    return results

# Step 2: Convert the generated outputs to 3D structure and visualize
def structure_from_generated_output(generated_results):
    """
    Convert generated model output to 3D protein structure

    Args:
        generated_results: Output from generate_from_model

    Returns:
        3D coordinates and visualization figure
    """
    # Extract results
    sequence = generated_results['protein_sequence']
    adjacency = generated_results['adjacency_matrix']

    # Convert adjacency to binary contact map with threshold
    binary_adjacency = (adjacency > 0.065).astype(np.float32)

    # Define distance constraints
    configs = [{'lower': 0, 'upper': 8.0}]

    # Generate 3D coordinates and visualization
    coords, fig = reconstruct_coords_local_plural_maps_colored(
        [binary_adjacency],  # List of contact maps
        configs,             # Distance constraints
        sequence,            # Amino acid sequence
        max_iter=2000,       # Optimization steps
        lr=0.01              # Learning rate
    )

    return coords, fig

# Complete usage example:
def main_visualization_pipeline():
    # Parameters
    parent_folder = "nanos_networkx_small"  # Update this to your data path
    chunk_length = 150
    max_proteins = 1000  # Limit number of proteins for faster execution

    # Set device
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load protein graph data directly
    try:
        full_graphs, full_sequences, subgraphs, subsequences = load_protein_graph_data(
            parent_folder, max_proteins, chunk_length
        )
    except Exception as e:
        print(f"Error loading graph data: {e}")
        print("Current directory contains:", os.listdir())
        return

    # First graph debug
    print("First few nodes of first graph:")
    first_graph = full_graphs[0]
    for i, node in enumerate(sorted(first_graph.nodes())[:5]):
        print(f"Node {node} attributes: {first_graph.nodes[node]}")

    # Define a standard set of amino acids (all 20 standard ones)
    STANDARD_AA = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
                   'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'X']

    # Get unique amino acids from sequences but ensure we have at least the standard 20
    UNIQUE_AA = set()
    for seq in full_sequences:
        UNIQUE_AA.update(seq)

    # Merge with standard AAs
    UNIQUE_AA = sorted(list(set(UNIQUE_AA).union(set(STANDARD_AA))))
    print(f"Unique amino acids: {len(UNIQUE_AA)}")
    print(f"Amino acids found: {', '.join(UNIQUE_AA)}")

    # Prepare data for training using graph data
    aa_sequences, adjacency_matrices, node_features = prepare_graph_data_for_training(
        subgraphs, subsequences, UNIQUE_AA
    )
    # Paths and settings
    model_path = "dual_output_gran_model.pt"
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

    # Sample input from validation set (or any other source)
    sample_adj = torch.tensor(adjacency_matrices[0], dtype=torch.float32)
    sample_features = torch.tensor(node_features[0], dtype=torch.float32)

    # Step 1: Generate from model
    generated_results = generate_from_model(
        model_path, sample_adj, sample_features, device, UNIQUE_AA
    )

    # Step 2: Convert to 3D structure
    coords, fig = structure_from_generated_output(generated_results)

    # Show visualization
    fig.show()

    # Optional: Save visualization
    fig.write_html("protein_structure.html")

    return coords, fig, generated_results

# Run the pipeline
coords, fig, generated_results = main_visualization_pipeline()

In [ ]:
def visualize_original_vs_generated(original_seq, original_adj, generated_seq, generated_adj):
    """
    Create a side-by-side visualization of original and generated protein structures
    """
    # Convert adjacency matrices to binary contact maps if needed
    binary_orig_adj = (original_adj > 0.5).astype(np.float32)
    binary_gen_adj = (generated_adj > 0.065).astype(np.float32)  # Adjust threshold as needed

    # Define distance constraints
    configs = [{'lower': 0, 'upper': 8.0}]

    # Generate 3D coordinates for both structures
    orig_coords, _ = reconstruct_coords_local_plural_maps_colored(
        [binary_orig_adj], configs, original_seq, max_iter=2000, lr=0.01
    )

    gen_coords, _ = reconstruct_coords_local_plural_maps_colored(
        [binary_gen_adj], configs, generated_seq, max_iter=2000, lr=0.01
    )

    # Create combined visualization
    fig = go.Figure()

    # Generate color mappings for amino acids
    def get_aa_colors(sequence):
        aa_categories = {
            # Simplified categories for visual distinction
            'A': 'hydrophobic', 'V': 'hydrophobic', 'L': 'hydrophobic',
            'I': 'hydrophobic', 'F': 'hydrophobic', 'W': 'hydrophobic',
            'M': 'hydrophobic', 'P': 'hydrophobic',
            'G': 'polar', 'S': 'polar', 'T': 'polar', 'C': 'polar',
            'Y': 'polar', 'N': 'polar', 'Q': 'polar',
            'K': 'positive', 'R': 'positive', 'H': 'positive',
            'D': 'negative', 'E': 'negative',
        }

        color_dict = {
            'hydrophobic': 'blue',
            'polar': 'green',
            'positive': 'red',
            'negative': 'orange',
            'other': 'purple'
        }

        return [color_dict.get(aa_categories.get(aa, 'other'), 'purple') for aa in sequence]

    orig_colors = get_aa_colors(original_seq)
    gen_colors = get_aa_colors(generated_seq)

    # Add original structure (left side)
    fig.add_trace(go.Scatter3d(
        x=orig_coords[:, 0],
        y=orig_coords[:, 1],
        z=orig_coords[:, 2],
        mode='markers+lines',
        marker=dict(
            size=5,
            color=orig_colors,
            opacity=0.8
        ),
        line=dict(
            color='lightblue',
            width=2
        ),
        text=[f"Original {i+1}: {aa}" for i, aa in enumerate(original_seq)],
        hoverinfo='text',
        name='Original',
        scene='scene1'
    ))

    # Add generated structure (right side)
    fig.add_trace(go.Scatter3d(
        x=gen_coords[:, 0],
        y=gen_coords[:, 1],
        z=gen_coords[:, 2],
        mode='markers+lines',
        marker=dict(
            size=5,
            color=gen_colors,
            opacity=0.8
        ),
        line=dict(
            color='lightgreen',
            width=2
        ),
        text=[f"Generated {i+1}: {aa}" for i, aa in enumerate(generated_seq)],
        hoverinfo='text',
        name='Generated',
        scene='scene2'
    ))

    # Configure layout with two 3D scenes
    fig.update_layout(
        title="Original vs Generated Protein Structure",
        scene1=dict(
            domain=dict(x=[0, 0.5], y=[0, 1]),
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.2, y=1.2, z=1.2))
        ),
        scene2=dict(
            domain=dict(x=[0.5, 1], y=[0, 1]),
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.2, y=1.2, z=1.2))
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        paper_bgcolor='white',
        showlegend=False,
    )

    return fig








In [ ]:
# Parameters
parent_folder = "nanos_networkx_small"  # Update this to your data path
chunk_length = 150
max_proteins = 1000  # Limit number of proteins for faster execution

# Set device
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# Load protein graph data directly
try:
    full_graphs, full_sequences, subgraphs, subsequences = load_protein_graph_data(
        parent_folder, max_proteins, chunk_length
    )
except Exception as e:
    print(f"Error loading graph data: {e}")
    print("Current directory contains:", os.listdir())


# First graph debug
print("First few nodes of first graph:")
first_graph = full_graphs[0]
for i, node in enumerate(sorted(first_graph.nodes())[:5]):
    print(f"Node {node} attributes: {first_graph.nodes[node]}")

# Define a standard set of amino acids (all 20 standard ones)
STANDARD_AA = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
               'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'X']

# Get unique amino acids from sequences but ensure we have at least the standard 20
UNIQUE_AA = set()
for seq in full_sequences:
    UNIQUE_AA.update(seq)

# Merge with standard AAs
UNIQUE_AA = sorted(list(set(UNIQUE_AA).union(set(STANDARD_AA))))
print(f"Unique amino acids: {len(UNIQUE_AA)}")
print(f"Amino acids found: {', '.join(UNIQUE_AA)}")

# Prepare data for training using graph data
aa_sequences, adjacency_matrices, node_features = prepare_graph_data_for_training(
    subgraphs, subsequences, UNIQUE_AA
)
# Paths and settings
model_path = "dual_output_gran_model.pt"
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Sample input from validation set (or any other source)
sample_adj = torch.tensor(adjacency_matrices[0], dtype=torch.float32)
sample_features = torch.tensor(node_features[0], dtype=torch.float32)

# Step 1: Generate from model
generated_results = generate_from_model(
    model_path, sample_adj, sample_features, device, UNIQUE_AA
)


In [ ]:
# Generate using a different sample (change 0 to any index)
sample_index = random.randint(0, len(adjacency_matrices)-1)  # You can use random.randint(0, len(adjacency_matrices)-1) for a random choice
print(sample_index)
# Sample input from a different protein
sample_adj = torch.tensor(adjacency_matrices[sample_index], dtype=torch.float32)
sample_features = torch.tensor(node_features[sample_index], dtype=torch.float32)

# Step 1: Generate from model
generated_results = generate_from_model(
    model_path, sample_adj, sample_features, device, UNIQUE_AA
)

# Get the original data
original_seq = subsequences[sample_index]
original_adj = adjacency_matrices[sample_index].numpy()

# Get the generated data
generated_seq = generated_results['protein_sequence']
generated_adj = generated_results['adjacency_matrix']

# Visualize comparison
fig = visualize_original_vs_generated(
    original_seq=original_seq,
    original_adj=original_adj,
    generated_seq=generated_seq,
    generated_adj=generated_adj
)
fig.show()

In [ ]:
def visualize_original_vs_generated(original_seq, original_adj, generated_seq, generated_adj):
    """
    Create a side-by-side visualization of original and generated protein structures
    with improved amino acid coloring
    """
    # Convert adjacency matrices to binary contact maps if needed
    binary_orig_adj = (original_adj > 0.5).astype(np.float32)
    binary_gen_adj = (generated_adj > 0.065).astype(np.float32)  # Adjust threshold as needed

    # Define distance constraints
    configs = [{'lower': 0, 'upper': 8.0}]

    # Detailed amino acid color scheme based on physicochemical properties
    aa_colors = {
        # Hydrophobic (blue shades)
        'A': '#0000FF',  # Blue
        'V': '#000080',  # Navy
        'L': '#4169E1',  # Royal Blue
        'I': '#1E90FF',  # Dodger Blue
        'M': '#00BFFF',  # Deep Sky Blue
        'F': '#87CEEB',  # Sky Blue
        'W': '#B0C4DE',  # Light Steel Blue

        # Polar (green shades)
        'S': '#008000',  # Green
        'T': '#006400',  # Dark Green
        'N': '#32CD32',  # Lime Green
        'Q': '#00FF00',  # Lime
        'Y': '#98FB98',  # Pale Green
        'C': '#90EE90',  # Light Green
        'G': '#ADFF2F',  # Green Yellow

        # Positively charged (red shades)
        'K': '#FF0000',  # Red
        'R': '#B22222',  # Fire Brick
        'H': '#FF6347',  # Tomato

        # Negatively charged (orange/yellow shades)
        'D': '#FFA500',  # Orange
        'E': '#FFD700',  # Gold

        # Special
        'P': '#800080',  # Purple
        'X': '#A9A9A9',  # Dark Gray
    }

    # Convert 3-letter AA codes to 1-letter if needed
    three_to_one = {
        'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
        'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
        'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
        'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V',
        'X': 'X', 'UNK': 'X'
    }

    # Process original sequence
    if len(original_seq) > 0 and len(original_seq[0]) == 3:  # 3-letter code
        orig_seq_1letter = [three_to_one.get(aa, 'X') for aa in original_seq]
    else:  # Already 1-letter code
        orig_seq_1letter = original_seq

    # Generate 3D coordinates for both structures
    print("Generating 3D structure for original protein...")
    orig_coords_results, orig_fig = reconstruct_coords_local_plural_maps_colored(
        [binary_orig_adj], configs, original_seq, hard_mask=None, sharpness=15.,
        window=25, dim=3, lr=1e-2, max_iter=2000, prints=5
    )

    print("Generating 3D structure for model-generated protein...")
    gen_coords_results, gen_fig = reconstruct_coords_local_plural_maps_colored(
        [binary_gen_adj], configs, generated_seq, hard_mask=None, sharpness=15.,
        window=25, dim=3, lr=1e-2, max_iter=2000, prints=5
    )

    # Use the coordinates from the results
    orig_coords = orig_coords_results
    gen_coords = gen_coords_results

    # Create combined visualization
    fig = go.Figure()

    # Add original structure (left side)
    # Nodes (amino acids)
    orig_colors = [aa_colors.get(aa, '#808080') for aa in orig_seq_1letter]

    fig.add_trace(go.Scatter3d(
        x=orig_coords[:, 0],
        y=orig_coords[:, 1],
        z=orig_coords[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=orig_colors,
            opacity=0.8
        ),
        text=[f"Orig {i+1}: {aa}" for i, aa in enumerate(orig_seq_1letter)],
        hoverinfo='text',
        name='Original',
        scene='scene1'
    ))

    # Lines connecting sequential residues
    x_lines, y_lines, z_lines = [], [], []
    line_colors = []

    for i in range(len(orig_coords)-1):
        # Add line segments
        x_lines.extend([orig_coords[i, 0], orig_coords[i+1, 0], None])
        y_lines.extend([orig_coords[i, 1], orig_coords[i+1, 1], None])
        z_lines.extend([orig_coords[i, 2], orig_coords[i+1, 2], None])
        line_colors.extend([orig_colors[i], orig_colors[i+1], 'rgba(0,0,0,0)'])

    fig.add_trace(go.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode='lines',
        line=dict(
            color='silver',
            width=2
        ),
        hoverinfo='none',
        showlegend=False,
        scene='scene1'
    ))

    # Add generated structure (right side)
    # Process generated sequence
    gen_colors = [aa_colors.get(aa, '#808080') for aa in generated_seq]

    fig.add_trace(go.Scatter3d(
        x=gen_coords[:, 0],
        y=gen_coords[:, 1],
        z=gen_coords[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=gen_colors,
            opacity=0.8
        ),
        text=[f"Gen {i+1}: {aa}" for i, aa in enumerate(generated_seq)],
        hoverinfo='text',
        name='Generated',
        scene='scene2'
    ))

    # Lines for generated structure
    x_lines, y_lines, z_lines = [], [], []

    for i in range(len(gen_coords)-1):
        # Add line segments
        x_lines.extend([gen_coords[i, 0], gen_coords[i+1, 0], None])
        y_lines.extend([gen_coords[i, 1], gen_coords[i+1, 1], None])
        z_lines.extend([gen_coords[i, 2], gen_coords[i+1, 2], None])

    fig.add_trace(go.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode='lines',
        line=dict(
            color='silver',
            width=2
        ),
        hoverinfo='none',
        showlegend=False,
        scene='scene2'
    ))

    # Add a legend for amino acid types
    amino_groups = {
        'Hydrophobic': ['A', 'V', 'L', 'I', 'M', 'F', 'W'],
        'Polar': ['S', 'T', 'N', 'Q', 'Y', 'C', 'G'],
        'Positively charged': ['K', 'R', 'H'],
        'Negatively charged': ['D', 'E'],
        'Special': ['P'],
        'Unknown': ['X']
    }

    # Add legend traces (invisible points with the right colors)
    for group_name, aas in amino_groups.items():
        for aa in aas:
            fig.add_trace(go.Scatter3d(
                x=[None], y=[None], z=[None],
                mode='markers',
                marker=dict(
                    size=6,
                    color=aa_colors.get(aa, '#808080'),
                    opacity=0.8
                ),
                name=f"{group_name}: {aa}",
                scene='scene1'  # Associate with first scene
            ))

    # Configure layout with two 3D scenes
    fig.update_layout(
        title="Original vs Generated Protein Structure",
        scene1=dict(
            domain=dict(x=[0, 0.5], y=[0, 1]),
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.2, y=1.2, z=1.2))
        ),
        scene2=dict(
            domain=dict(x=[0.5, 1], y=[0, 1]),
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='cube',
            camera=dict(eye=dict(x=1.2, y=1.2, z=1.2))
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        paper_bgcolor='white',
        legend=dict(
            x=0.01,
            y=0.99,
            traceorder='normal',
            bgcolor='rgba(255,255,255,0.5)',
            bordercolor='rgba(0,0,0,0.5)',
            borderwidth=1
        )
    )

    # Add sequence information as annotations
    fig.add_annotation(
        x=0.25, y=1.05,
        xref='paper', yref='paper',
        text=f"Original: {original_seq[:15]}..." if len(original_seq) > 15 else original_seq,
        showarrow=False,
        font=dict(size=10)
    )

    fig.add_annotation(
        x=0.75, y=1.05,
        xref='paper', yref='paper',
        text=f"Generated: {generated_seq[:15]}..." if len(generated_seq) > 15 else generated_seq,
        showarrow=False,
        font=dict(size=10)
    )

    return fig

In [ ]:
# Generate using a different sample (change 0 to any index)
sample_index = random.randint(0, len(adjacency_matrices)-1)  # You can use random.randint(0, len(adjacency_matrices)-1) for a random choice
print(sample_index)
# Sample input from a different protein
sample_adj = torch.tensor(adjacency_matrices[sample_index], dtype=torch.float32)
sample_features = torch.tensor(node_features[sample_index], dtype=torch.float32)

# Step 1: Generate from model
generated_results = generate_from_model(
    model_path, sample_adj, sample_features, device, UNIQUE_AA
)

# Get the original data
original_seq = subsequences[sample_index]
original_adj = adjacency_matrices[sample_index].numpy()

# Get the generated data
generated_seq = generated_results['protein_sequence']
generated_adj = generated_results['adjacency_matrix']

# Visualize comparison
fig = visualize_original_vs_generated(
    original_seq=original_seq,
    original_adj=original_adj,
    generated_seq=generated_seq,
    generated_adj=generated_adj
)
fig.show()